In [0]:
#************************************************************************************************************************************
#*                                                                                                                                  *
#*   NOTEBOOK:     Auto_Run_EDW_Data_Load.                                                                                       *
#*                                                                                                                                  *
#*   DESCRIPTION:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT PARMS:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT FILES:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   OUTPUT FILE:                                                                                                                   *
#*                                                                                                                                  *
#*   EXITS:       0 - success                                                                                                       *
#*                <> 0 - failure                                                                                                    *
#*                                                                                                                                  *
#************************************************************************************************************************************
#*                                                                                                                                  *
#*                                                 Modification Log                                                                 *
#*                                                                                                                                  *
#*    Date     CO                 Author              Description                                                                   *
#* ---------- ------------------  -----------------   ------------------------------------------------------------------------------*
#* 05/06/2025 000000000000000000  Ahmad Afzaal        Initial Release.                                                              *
#************************************************************************************************************************************


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("base_path", "s3a://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")
dbutils.widgets.text("log_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Ref")
dbutils.widgets.text("delta_table", "oh_apm_stg.vendor_extracts.load_report_log_cpc_Stg_Ref") #temp tables Staging 
dbutils.widgets.text("copy_target_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/process/ETL_Ref")
dbutils.widgets.text("ctl_file_identifier", "ODM.EDW.VEN.REFERENCE.WEEKLY")
dbutils.widgets.text("columns_to_check_for_nulls", "SAK_RECIP")  # comma-separated

In [0]:
base_path = dbutils.widgets.get("base_path")
log_table = dbutils.widgets.get("log_table")
delta_table = dbutils.widgets.get("delta_table")
copy_target_path = dbutils.widgets.get("copy_target_path")
ctl_file_identifier = dbutils.widgets.get("ctl_file_identifier")
columns_to_check_for_nulls = dbutils.widgets.get("columns_to_check_for_nulls").split(",")


In [0]:
import os
import json
import re
from datetime import datetime
from pyspark.sql import functions as f
from pyspark.dbutils import DBUtils
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql import SparkSession
from pyspark.sql import Row
# Initialize Spark session
spark = SparkSession.builder.appName("CTL-GZ-Reconciliation").getOrCreate()

# Get all folders under base_path
all_folders = [f.path for f in dbutils.fs.ls(base_path) if f.isDir()]

# Extract date from folder name
def extract_date_or_mtime(folder_path):
    match = re.search(r"dt=(\d{8})", folder_path)
    if match:
        return datetime.strptime(match.group(1), "%Y%m%d")
    match = re.search(r"dt=(\d{4}-\d{2}-\d{2})", folder_path)
    if match:
        return datetime.strptime(match.group(1), "%Y-%m-%d")

    # Fallback: Use latest file's modification time in the folder
    try:
        files = dbutils.fs.ls(folder_path)
        if files:
            return max(datetime.fromtimestamp(f.modificationTime / 1000) for f in files)
    except:
        return None
    return None

# Get folders with valid dates
folders_with_dates = [(f, extract_date_or_mtime(f)) for f in all_folders if extract_date_or_mtime(f)]
if not folders_with_dates:
    raise Exception("No dated folders found.")

# Find the latest date
latest_date = max(date for _, date in folders_with_dates)

# Filter folders with the latest date
latest_folders = [f for f, date in folders_with_dates if date == latest_date]
print(f"\n📁 Latest folder date: {latest_date.strftime('%Y-%m-%d')}")
for folder in latest_folders:
    print(f"➡️ Using folder: {folder}")
    
# Collect all files from the latest folders
all_files = []
for folder in latest_folders:
    all_files.extend(dbutils.fs.ls(folder))

# Find CTL files
ctl_files = [f for f in all_files if f.name.endswith(".ctl") and ctl_file_identifier in f.name]
if not ctl_files:
    raise Exception("❌ No .ctl files found in latest folders.")

# Load reconciliation log if exists
try:
    log_df = spark.read.table(log_table)
except:
    log_df = spark.createDataFrame([], schema="""
        CTL_File STRING, GZ_File STRING, Status STRING,
        Row_Count_in_CTL INT, Row_Count_in_GZ INT,
        Null_Columns_Failed STRING, Processed_Timestamp TIMESTAMP
    """)

# Filter out CTL files that were already successfully processed
successful_ctls = log_df.filter(f.col("Status") == "SUCCESS").select("CTL_File").distinct().rdd.flatMap(lambda x: x).collect()
ctl_files = [f for f in ctl_files if f.name not in successful_ctls]

if not ctl_files:
    raise Exception("❌ No new or failed .ctl files to process.")
    exit(1)
# Pick the latest unprocessed or failed CTL file
latest_ctl = sorted(ctl_files, key=lambda x: x.name, reverse=True)[0]
print(f"\nUsing CTL File: {latest_ctl.path}")

# Extract timestamp from CTL file name for Date_Received_by_APM
ctl_timestamp_match = re.search(r'(\d{8})', latest_ctl.name)
ctl_received_ts = datetime.fromtimestamp(latest_ctl.modificationTime / 1000).strftime("%Y-%m-%d %H:%M:%S")
ctl_received_date = datetime.fromtimestamp(latest_ctl.modificationTime / 1000).strftime("%Y-%m-%d")
# Pass to next task in workflow
dbutils.jobs.taskValues.set(key="ctl_received_date", value=ctl_received_date)

print(f".CTL recived date by APM:{ctl_received_date}")
# Read CTL file
ctl_df = spark.read.text(latest_ctl.path)
ctl_data = ctl_df.collect()

# Parse CTL entries
gz_expected_info = []
for row in ctl_data:
    parts = row[0].split('|')
    if len(parts) >= 2:
        gz_expected_info.append((parts[0], int(parts[1])))


gz_files_info = {f.name: f.path for f in all_files if f.name.endswith(".gz")}

missing_files = [gz_name for gz_name, _ in gz_expected_info if gz_name not in gz_files_info]
if missing_files:
    raise Exception(f"❌ Missing .gz files referenced in .ctl but not found: {', '.join(missing_files)}")

# Reconciliation
results = []
has_critical_failure = False
for gz_file_name, expected_count in gz_expected_info:
    matching_file_path = gz_files_info.get(gz_file_name)
    null_columns_failed = []
    error_count = None
    status = None
    actual_count = None

    if matching_file_path:
        try:
            # Check if file is non-empty
            if dbutils.fs.ls(matching_file_path):
                gz_df = spark.read.option("header", "true").option("inferSchema", "true").csv(matching_file_path)
                actual_count = gz_df.count()

                # Null check
                for col_name in columns_to_check_for_nulls:
                    if col_name in gz_df.columns:
                        null_count = gz_df.filter(col(col_name).isNull()).count()
                        if null_count > 0:
                            null_columns_failed.append(col_name)

                if null_columns_failed:
                    status = "NULL_VALUES_FOUND"
                    has_critical_failure = True
                    error_count = total_nulls
                elif actual_count != expected_count:
                    status = "Failed"
                    error_count = abs(actual_count - expected_count)
                else:
                    status = "SUCCESS"
            else:
                status = "EMPTY_FILE"
                actual_count = 0
                has_critical_failure = True
        except Exception as e:
            status = "CORRUPT"
            actual_count = None
            has_critical_failure = True
    else:
        status = "MISSING"
        actual_count = None
        has_critical_failure = True

    results.append(Row(
        CTL_File=latest_ctl.name,
        GZ_File=gz_file_name,
        Status=status,
        Row_Count_in_CTL=expected_count,
        Row_Count_in_GZ=actual_count,
        Null_Columns_Failed=",".join(null_columns_failed) if null_columns_failed else None,
        Number_of_Error_Records=error_count
    ))



# Display and log results
if results:
    result_df = spark.createDataFrame(results, schema="CTL_File STRING, GZ_File STRING, Status STRING, Row_Count_in_CTL INT, Row_Count_in_GZ INT, Null_Columns_Failed STRING, Number_of_Error_Records bigint")
    result_df = result_df.withColumn("Processed_Timestamp", f.current_timestamp())
    print("\nFinal Reconciliation Report:")
    result_df.show(truncate=False)

    # Append to Unity Catalog Delta table
    result_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(log_table)
if has_critical_failure:
    raise Exception("❌ One or more critical reconciliation failures detected. Stopping workflow.")

if results:
    schema = StructType([
        StructField("File_Name", StringType(), True),
        StructField("Date_Received_by_APM", StringType(), True),
        StructField("Status", StringType(), True),
        StructField("Number_of_Records_Received", LongType(), True),
        StructField("Number_of_Records_Loaded", LongType(), True),
        StructField("Number_of_Error_Records", LongType(), True),
        StructField("Start_Load_Date", StringType(), True),
        StructField("End_Load_Date", StringType(), True)
    ])

    data = [
        (
            r['GZ_File'],  # File_Name
            ctl_received_ts,  # Date_Received_by_APM
            r[2],  # Status
            r[3],  # Number_of_Records_Received
            r[4],  # Number_of_Records_Loaded
            r['Number_of_Error_Records'],
            None,  # Start_Load_Date
            None   # End_Load_Date
        )
        for r in results if r['Status'] not in ["NULL_VALUES_FOUND", "MISSING", "CORRUPT"]
    ]

if not has_critical_failure:
    if any(r['Status'] == "Failed" for r in results):
        print("⚠️ Non-critical failure detected. Writing 'Failed' status and stopping workflow.")
        if data:
            df = spark.createDataFrame(data, schema=schema)
            df.write.format("delta").mode("append").saveAsTable(delta_table)
            print("📄 Wrote 'Failed' status to Delta table.")
        raise Exception("❌ Pre‐Validation encountered a non-critical failure. Stopping workflow.")
    else:
        # All statuses are success
        print("✅ No critical failure and all statuses are 'Success'. Proceeding with data load and file operations.")
        if data:
            df = spark.createDataFrame(data, schema=schema)
            df.write.format("delta").mode("append").saveAsTable(delta_table)
            print("📄 Uploaded to Delta table.")

        # Clean folder
        if not copy_target_path.endswith("/"):
            copy_target_path += "/"

        def folder_exists(path):
            try:
                _ = dbutils.fs.ls(path)
                return True
            except Exception:
                return False

        if folder_exists(copy_target_path):
            files_in_target = dbutils.fs.ls(copy_target_path)
            for f in files_in_target:
                try:
                    dbutils.fs.rm(f.path, recurse=True)
                except Exception as e:
                    print(f"❌ Failed to delete {f.path}: {e}")
            print(f"🧹 Cleared files inside {copy_target_path} (folder preserved)")
        else:
            print(f"📁 Folder {copy_target_path} does not exist, skipping deletion step")

        # Copy CTL
        try:
            dbutils.fs.cp(latest_ctl.path, copy_target_path + latest_ctl.name)
            print(f"📄 Copied CTL file: {latest_ctl.name}")
        except Exception as e:
            raise Exception(f"❌ Failed to copy CTL file: {e}")

        # Copy GZ files
        for gz_file_name, _ in gz_expected_info:
            gz_path = gz_files_info.get(gz_file_name)
            if gz_path:
                try:
                    dbutils.fs.cp(gz_path, copy_target_path + gz_file_name)
                    print(f"📦 Copied GZ file: {gz_file_name}")
                except Exception as e:
                    raise Exception(f"❌ Failed to copy GZ file: {gz_file_name}, error: {e}")

        # # Extract and return only the date portion from ctl_received_ts
        # dbutils.notebook.exit(json.dumps({"received_date": ctl_received_date}))
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            run_id = ctx.tags().get("runId")

            # Convert to string before storing
            if run_id:
                dbutils.jobs.taskValues.set(key="run_id", value=str(run_id))
            else:
                raise ValueError("❌ run_id not found in context. This notebook must be run as part of a job.")

else:
    print("⛔ Critical failure detected. Skipping all operations.")